# 04 — CNN (EEGNet) on Colab: LOSO x 5 seeds (Weeks 3-4)

Reproduces the Week 2 CNN pipeline (`src/models/cnn.py`) on a GPU runtime and
runs the full **leave-one-subject-out (LOSO)** protocol with 5 model seeds —
the run that is too slow for the local VM.

**Protocol notes**
- Data are rebuilt from the official FatigueSet download under
  `docs/label_rules.md` **Amendment 1** (frontal-gated retention, temporal
  masking with explicit indicator channels).
- LOSO early stopping uses **subject-grouped** validation (one whole
  training participant held out) — feasible here because each LOSO training
  fold has 11 participants.
- ⚠️ The *within-subject* CNN protocol's session-grouped early stopping was
  found structurally infeasible for most participants in Week 2 (after the
  M1 cap most participants have 2 usable sessions; holding out the test
  session leaves a single training session, so a session-grouped validation
  split leaves nothing to train on). It is **not** run here pending a
  documented amendment decision. See the Week 2 project lead report.
- Results land in `experiments/runs/` per `experiments/README.md`; commit
  them back to GitHub from the last cell.

In [ ]:
# 1. Get the code (private repo — authenticate when prompted)
!git clone https://github.com/visionbyangelic/generalizable-fatigue-detection.git
%cd generalizable-fatigue-detection

In [ ]:
# 2. Get the dataset from the OFFICIAL FatigueSet Drive link (never a mirror)
!pip -q install gdown
!mkdir -p data/raw
!gdown --id 1i-LPuF6j91T77tAjmZ7TqZfYYPBlTE5_ -O data/raw/fatigueset.zip
!unzip -q -o data/raw/fatigueset.zip -d data/raw/
!ls data/raw

In [ ]:
# 3. Dependencies (Colab ships torch; install the rest)
!pip -q install -r requirements.txt
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# 4. Rebuild the Amendment-1 window dataset + mandatory diagnostics
!python src/preprocessing/build_windows_a1.py
!python src/preprocessing/diagnostics_a1.py

In [ ]:
# 5. Full LOSO run: EEGNet, model seeds 0-4 (cap seed 0 = canonical set)
!python src/models/cnn.py --protocol loso --seeds 0,1,2,3,4 --cap-seed 0

In [ ]:
# 6. (Optional) LOSO seed-variance across cap seeds 1-4
# !python src/models/cnn.py --protocol loso --seeds 0,1,2,3,4 --cap-seed 1

In [ ]:
# 7. Inspect results, then commit them back (fill in your git identity)
!ls experiments/runs/ | tail -5
!cat results/tables/cnn_loso_subjective_a1_summary.csv
# !git config user.email "you@example.com"
# !git config user.name "Angelic Charles"
# !git add experiments results && git commit -m "Colab: CNN LOSO x5 seeds" && git push